# Trajectory Verification Test

Run this notebook to verify trajectories are working.

In [ ]:
# Force reload of visualization module
import importlib
import sys

# Clear any cached imports
if 'great_silence.visualization.interactive_3d' in sys.modules:
    importlib.reload(sys.modules['great_silence.visualization.interactive_3d'])

from great_silence.notebook.helpers import reconstruct_simulation_from_hdf5
from great_silence.visualization.interactive_3d import Interactive3DVisualizer

print("✓ Modules loaded (fresh)")

In [ ]:
# Load simulation
sim = reconstruct_simulation_from_hdf5('output/my_simulation.h5')

expanding_civs = [c for c in sim.civilizations if len(c.colonized_stars) > 1]
print(f"Loaded simulation:")
print(f"  Snapshots: {len(sim.snapshots)}")
print(f"  Civilizations: {len(sim.civilizations)}")
print(f"  Expanding: {len(expanding_civs)}")

if len(expanding_civs) == 0:
    print("\n⚠ No expanding civilizations - trajectories will be empty")
else:
    print(f"\n✓ {len(expanding_civs)} expanding civilizations found")

In [ ]:
# Create visualization with trajectories
print("Creating animated figure with trajectories...")

viz = Interactive3DVisualizer(sim)
fig = viz.create_animated_figure(
    subsample_stars=1000,
    show_stars=True,
    show_hazards=False,  # Disable hazards for cleaner view
    show_trajectories=True,  # ← TRAJECTORIES ENABLED
    show_spheres=False,
    show_probes=True,
    interpolation_factor=3
)

print(f"✓ Figure created with {len(fig.frames)} frames")

# Count trajectory traces
traj_count = 0
for frame in fig.frames:
    traj_count += sum(1 for t in frame.data 
                     if hasattr(t, 'name') and t.name and 'Civ' in str(t.name)
                     and hasattr(t, 'mode') and t.mode == 'lines')

print(f"  Total trajectory traces across all frames: {traj_count}")

if traj_count == 0:
    print("\n❌ NO TRAJECTORIES IN FRAMES!")
    print("   Problem with trajectory builder")
else:
    print(f"\n✓ {traj_count} trajectory traces found")
    print("   Trajectories should be visible in animation")

In [ ]:
# Display figure
fig.update_layout(
    title="Civilization Expansion - WITH TRAJECTORIES",
    scene=dict(
        camera=dict(
            eye=dict(x=1.5, y=1.5, z=1.5)  # Good viewing angle
        )
    )
)

fig.show()

print("\n" + "="*70)
print("WHAT TO LOOK FOR:")
print("="*70)
print("1. Use the animation slider at the bottom")
print("2. Move to frame 5+ (around 2000 Myr)")
print("3. Look for COLORED LINES connecting stars")
print("4. Lines should be blue/colored (Kardashev-level mapped)")
print("5. Lines grow over time as colonies arrive")
print("\nFrame 0-1: No trajectories (too early)")
print("Frame 2+: Trajectories start appearing")
print("="*70)

In [ ]:
# Save to HTML for comparison
output_path = 'output/trajectory_verification.html'
fig.write_html(output_path)
print(f"✓ Saved to {output_path}")
print(f"  Open in web browser to verify trajectories")